# 🤖 Question Answering API
Model: `deepset/roberta-base-squad2`

## Bước 1: Cài đặt thư viện

In [ ]:
!pip install fastapi uvicorn transformers==4.44.0 torch pyngrok requests nest_asyncio "numpy<2.0"

## Bước 2: Viết file main.py

In [ ]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from transformers import pipeline

app = FastAPI(
    title="Question Answering API",
    description="API để trả lời câu hỏi dựa trên đoạn văn bản cho trước, sử dụng model deepset/roberta-base-squad2.",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

class QARequest(BaseModel):
    context: str
    question: str

@app.get("/")
def root():
    return {
        "name": "Question Answering API",
        "description": "Nhận vào một đoạn văn bản (context) và một câu hỏi (question), trả về câu trả lời được trích xuất từ đoạn văn.",
        "model": "deepset/roberta-base-squad2",
        "endpoints": {
            "GET  /": "Thông tin API",
            "GET  /health": "Kiểm tra trạng thái hệ thống",
            "POST /predict": "Trả lời câu hỏi từ đoạn văn bản"
        }
    }

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": qa_pipeline is not None}

@app.post("/predict")
def predict(body: QARequest):
    if not body.context or not body.context.strip():
        raise HTTPException(status_code=400, detail="'context' không được để trống.")
    if not body.question or not body.question.strip():
        raise HTTPException(status_code=400, detail="'question' không được để trống.")
    if len(body.context) > 5000:
        raise HTTPException(status_code=400, detail="'context' không được vượt quá 5000 ký tự.")
    try:
        result = qa_pipeline(question=body.question, context=body.context)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Lỗi khi chạy model: {str(e)}")
    return {
        "answer": result["answer"],
        "score": round(result["score"], 4),
        "start": result["start"],
        "end": result["end"]
    }

## Bước 3: Chạy server FastAPI

In [ ]:
import nest_asyncio
import uvicorn
import threading
nest_asyncio.apply()

from main import app

def run():
    uvicorn.run(app, host='0.0.0.0', port=8000)

t = threading.Thread(target=run, daemon=True)
t.start()
print('Server đang chạy tại http://0.0.0.0:8000')

## Bước 4: Tạo public URL bằng pyngrok

In [ ]:
from pyngrok import ngrok

# Điền token của bạn tại https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token('YOUR_NGROK_TOKEN_HERE')

public_url = ngrok.connect(8000)
print(f'Public URL: {public_url}')

## Bước 5: Test API

In [ ]:
import requests

API_URL = 'http://127.0.0.1:8000'

# Test 1: GET /
print('GET /')
print(requests.get(f'{API_URL}/').json())

# Test 2: GET /health
print('\nGET /health')
print(requests.get(f'{API_URL}/health').json())

# Test 3: POST /predict
print('\nPOST /predict - Test 1')
r = requests.post(f'{API_URL}/predict', json={
    'context': 'The Eiffel Tower was built in 1889 in Paris, France.',
    'question': 'Where is the Eiffel Tower?'
})
print(r.json())

# Test 4: POST /predict
print('\nPOST /predict - Test 2')
r = requests.post(f'{API_URL}/predict', json={
    'context': 'Python is a high-level programming language created by Guido van Rossum in 1991.',
    'question': 'Who created Python?'
})
print(r.json())